# FT-Transformer

In [15]:
!python -m pip install --upgrade pip setuptools wheel

Defaulting to user installation because normal site-packages is not writeable
Could not fetch URL https://pypi.org/simple/pip/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/pip/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping
Could not fetch URL https://pypi.org/simple/setuptools/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/setuptools/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping
Could not fetch URL https://pypi.org/simple/wheel/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/wheel/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping


In [16]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Could not fetch URL https://pypi.org/simple/ipywidgets/: There was a problem confirming the ssl certificate: HTTPSConnectionPool(host='pypi.org', port=443): Max retries exceeded with url: /simple/ipywidgets/ (Caused by SSLError(SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:1129)'))) - skipping


ERROR: Could not find a version that satisfies the requirement ipywidgets (from versions: none)
ERROR: No matching distribution found for ipywidgets


In [17]:
!pip install rtdl_revisiting_models -q

In [18]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from rtdl_revisiting_models import FTTransformer

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = r"D:\墨大sml作业\FeatureA_Repeated"
OUTPUT_PATH = r"D:\墨大sml作业\Official_FTTransformer_FeatureA_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cpu


In [20]:
feature_cols = [
    "user_avg_rating",
    "user_rating_count",
    "user_rating_std",
    "user_like_count",
    "user_like_ratio",
    "user_rating_timespan",
    "user_avg_gap_days",
    "item_avg_rating",
    "item_rating_count",
    "item_rating_std",
    "item_like_count",
    "item_like_ratio",
    "global_mean",
    "movie_age_at_rating"
]

In [21]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [22]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [23]:
def stratified_sample_binary(df, sample_size, random_seed):
    pos_df = df[df["label"] == 1]
    neg_df = df[df["label"] == 0]

    pos_n = sample_size // 2
    neg_n = sample_size - pos_n

    pos_sample = pos_df.sample(
        n=pos_n,
        random_state=random_seed
    )

    neg_sample = neg_df.sample(
        n=neg_n,
        random_state=random_seed
    )

    sampled_df = pd.concat(
        [pos_sample, neg_sample],
        axis=0
    ).sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return sampled_df

In [24]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [25]:
TRAIN_SAMPLE_SIZE = 200000
TEST_SAMPLE_SIZE = 100000

LR_VALUES = [1e-4, 5e-5]
WEIGHT_DECAY_VALUES = [1e-5]

BATCH_SIZE = 4096
EPOCHS = 3

all_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Repeat {repeat_id}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_A_train.csv")
    test_path = os.path.join(repeat_folder, "feature_A_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Sampled train shape:", train_df.shape)
    print("Sampled test shape:", test_df.shape)

    best_result = None
    best_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:
            print(f"Trying lr={lr}, weight_decay={weight_decay}")

            metrics = train_official_ft_transformer(
                train_df=train_df,
                test_df=test_df,
                lr=lr,
                weight_decay=weight_decay,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS
            )

            if metrics["f1"] > best_f1:
                best_f1 = metrics["f1"]

                best_result = {
                    "repeat": repeat_id,
                    "best_lr": lr,
                    "best_weight_decay": weight_decay,
                    "batch_size": BATCH_SIZE,
                    **metrics
                }

    all_results.append(best_result)
    print(best_result)

Repeat 1
Sampled train shape: (200000, 17)
Sampled test shape: (100000, 17)
Trying lr=0.0001, weight_decay=1e-05
Epoch 1 loss: 27.7309
Epoch 2 loss: 26.8954
Epoch 3 loss: 26.7692
Trying lr=5e-05, weight_decay=1e-05
Epoch 1 loss: 28.1112
Epoch 2 loss: 26.9911
Epoch 3 loss: 26.8878
{'repeat': 1, 'best_lr': 5e-05, 'best_weight_decay': 1e-05, 'batch_size': 4096, 'accuracy': np.float64(0.71476), 'precision': np.float64(0.7044554455445544), 'recall': np.float64(0.73996), 'f1': np.float64(0.7217713616855247), 'auc': np.float64(0.7893428271999694), 'tp': np.int64(36998), 'tn': np.int64(34478), 'fp': np.int64(15522), 'fn': np.int64(13002)}
Repeat 2
Sampled train shape: (200000, 17)
Sampled test shape: (100000, 17)
Trying lr=0.0001, weight_decay=1e-05
Epoch 1 loss: 28.1839
Epoch 2 loss: 26.9466
Epoch 3 loss: 26.8323
Trying lr=5e-05, weight_decay=1e-05
Epoch 1 loss: 28.0292
Epoch 2 loss: 27.0069
Epoch 3 loss: 26.884
{'repeat': 2, 'best_lr': 0.0001, 'best_weight_decay': 1e-05, 'batch_size': 4096, 

In [26]:
results_df = pd.DataFrame(all_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "Official_FTTransformer_results.csv"
)

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:")
print(results_path)

results_df

Saved to:
D:\墨大sml作业\Official_FTTransformer_FeatureA_Results\Official_FTTransformer_results.csv


,repeat,best_lr,best_weight_decay,batch_size,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.00005,0.00001,4096,0.71476,0.704455,0.73996,0.721771,0.789343,36998,34478,15522,13002
1,2,0.00010,0.00001,4096,0.71330,0.700342,0.74564,0.722281,0.786483,37282,34048,15952,12718
2,3,0.00010,0.00001,4096,0.71565,0.697521,0.76154,0.728126,0.790054,38077,33488,16512,11923
3,4,0.00005,0.00001,4096,0.71640,0.705782,0.74220,0.723533,0.790948,37110,34530,15470,12890
4,5,0.00005,0.00001,4096,0.71576,0.703516,0.74584,0.724060,0.790800,37292,34284,15716,12708
5,6,0.00010,0.00001,4096,0.71533,0.710765,0.72616,0.718380,0.789139,36308,35225,14775,13692
6,7,0.00010,0.00001,4096,0.71261,0.701240,0.74086,0.720506,0.788488,37043,34218,15782,12957
7,8,0.00005,0.00001,4096,0.71673,0.705451,0.74418,0.724298,0.790387,37209,34464,15536,12791
8,9,0.00005,0.00001,4096,0.71574,0.696477,0.76476,0.729023,0.791319,38238,33336,16664,11762
9,10,0.00005,0.00001,4096,0.71471,0.705319,0.73758,0.721089,0.789515,36879,34592,15408,13121


In [27]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "Official_FTTransformer_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.715099,0.001302,0.000412
1,precision,0.703087,0.004275,0.001352
2,recall,0.744872,0.011185,0.003537
3,f1,0.723307,0.003299,0.001043
4,auc,0.789648,0.001424,0.000450
